# 05E — Primary Model Freeze and Incident-Level Uncertainty Audit

**Build ID:** `05E_V1_PRIMARY_MODEL_FREEZE_INCIDENT_UNCERTAINTY`

05D did not improve the row-weighted full-extended model enough to justify
replacing the simpler fixed 05C baseline.

05E therefore performs **no model fitting**. It freezes the primary model as:

`05C_FIXED_ROW_RAW`

and quantifies uncertainty with a paired wildfire-cluster bootstrap.

Secondary comparison:
`05D_INCIDENT_EQUAL_NESTED_CALIBRATED`.

In [ ]:
import os  # Configurable project root for the public release.
NOTEBOOK_BUILD_ID = "05E_V1_PRIMARY_MODEL_FREEZE_INCIDENT_UNCERTAINTY"

from pathlib import Path
import sys, subprocess, importlib.util, json
from datetime import datetime, timezone

PROJECT_ROOT = Path(os.environ.get("FIRELINE_PROJECT_ROOT", "."))

BASELINE_ROW_PATH = (
    PROJECT_ROOT / "data" / "processed" / "baseline_modeling_v1" /
    "oof_predictions" / "E_FULL_EXTENDED__row.parquet"
)

NESTED_ROW_PATH = (
    PROJECT_ROOT / "data" / "processed" / "nested_tuning_calibration_v1" /
    "oof_predictions" / "E_FULL_EXTENDED__row__nested_tuned.parquet"
)

NESTED_IE_PATH = (
    PROJECT_ROOT / "data" / "processed" / "nested_tuning_calibration_v1" /
    "oof_predictions" / "E_FULL_EXTENDED__incident_equal__nested_tuned.parquet"
)

OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "primary_model_freeze_v1"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

for p in [BASELINE_ROW_PATH, NESTED_ROW_PATH, NESTED_IE_PATH]:
    assert p.exists(), p

EXPECTED_UNITS = 304_143
EXPECTED_INCIDENTS = 381
PRIMARY_MODEL_ID = "05C_FIXED_ROW_RAW"
BOOTSTRAP_REPS = 5000
BOOTSTRAP_SEED = 20260825
PRED_EPS = 1e-6

required = {
    "pandas": "pandas",
    "numpy": "numpy",
    "sklearn": "scikit-learn",
}

for module, package in required.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss, log_loss

pd.set_option("display.max_columns", 500)
pd.set_option("display.max_rows", 500)
pd.set_option("display.width", 420)

print("RUNNING:", NOTEBOOK_BUILD_ID)

## 1. Load and align saved OOF predictions

In [ ]:
base = pd.read_parquet(BASELINE_ROW_PATH)
row = pd.read_parquet(NESTED_ROW_PATH)
ie = pd.read_parquet(NESTED_IE_PATH)

for name, frame in [("base", base), ("row", row), ("ie", ie)]:
    assert len(frame) == EXPECTED_UNITS, name
    assert frame["unit_uid"].is_unique, name
    assert frame["_irwin_norm"].nunique() == EXPECTED_INCIDENTS, name

base = base[
    ["unit_uid", "_irwin_norm", "IncidentName", "Year", "hold_label", "outer_fold", "pred_prob"]
].rename(columns={"pred_prob": "05C_FIXED_ROW_RAW"})

row = row[
    ["unit_uid", "_irwin_norm", "hold_label", "outer_fold", "pred_prob_raw", "pred_prob_calibrated"]
].rename(columns={
    "hold_label": "hold_label_row",
    "outer_fold": "outer_fold_row",
    "pred_prob_raw": "05D_TUNED_ROW_RAW",
    "pred_prob_calibrated": "05D_TUNED_ROW_CAL",
})

ie = ie[
    ["unit_uid", "_irwin_norm", "hold_label", "outer_fold", "pred_prob_calibrated"]
].rename(columns={
    "hold_label": "hold_label_ie",
    "outer_fold": "outer_fold_ie",
    "pred_prob_calibrated": "05D_INCIDENT_EQUAL_CAL",
})

pred = (
    base
    .merge(row, on=["unit_uid", "_irwin_norm"], how="inner", validate="one_to_one")
    .merge(ie, on=["unit_uid", "_irwin_norm"], how="inner", validate="one_to_one")
)

assert len(pred) == EXPECTED_UNITS
assert (pred["hold_label"] == pred["hold_label_row"]).all()
assert (pred["hold_label"] == pred["hold_label_ie"]).all()
assert (pred["outer_fold"] == pred["outer_fold_row"]).all()
assert (pred["outer_fold"] == pred["outer_fold_ie"]).all()

pred = pred.drop(columns=[
    "hold_label_row", "hold_label_ie", "outer_fold_row", "outer_fold_ie"
])

MODEL_IDS = [
    "05C_FIXED_ROW_RAW",
    "05D_TUNED_ROW_RAW",
    "05D_TUNED_ROW_CAL",
    "05D_INCIDENT_EQUAL_CAL",
]

assert np.isfinite(pred[MODEL_IDS].to_numpy(float)).all()

print("Aligned:", pred.shape)

## 2. Point metrics

In [ ]:
def clip_prob(p):
    return np.clip(np.asarray(p, dtype=float), PRED_EPS, 1.0 - PRED_EPS)

def logloss_vec(y, p):
    y = np.asarray(y, dtype=float)
    p = clip_prob(p)
    return -(y * np.log(p) + (1.0 - y) * np.log(1.0 - p))

def incident_table(frame, model_id):
    rows = []
    for incident, g in frame.groupby("_irwin_norm", sort=False):
        y = g["hold_label"].to_numpy(int)
        p = g[model_id].to_numpy(float)
        mixed = np.unique(y).size == 2
        rows.append({
            "_irwin_norm": incident,
            "units": len(g),
            "brier_sum": float(np.sum((p-y)**2)),
            "logloss_sum": float(np.sum(logloss_vec(y,p))),
            "brier": float(np.mean((p-y)**2)),
            "log_loss": float(np.mean(logloss_vec(y,p))),
            "mixed_class": mixed,
            "pr_auc": float(average_precision_score(y,p)) if mixed else np.nan,
            "roc_auc": float(roc_auc_score(y,p)) if mixed else np.nan,
        })
    return pd.DataFrame(rows)

y_all = pred["hold_label"].to_numpy(int)
point_rows = []
incident_tables = {}

for model_id in MODEL_IDS:
    p = pred[model_id].to_numpy(float)
    inc = incident_table(pred, model_id)
    incident_tables[model_id] = inc
    mixed = inc["mixed_class"]

    point_rows.append({
        "model_id": model_id,
        "mean_pred": float(np.mean(p)),
        "pooled_pr_auc": float(average_precision_score(y_all,p)),
        "pooled_roc_auc": float(roc_auc_score(y_all,p)),
        "pooled_brier": float(brier_score_loss(y_all,p)),
        "pooled_log_loss": float(log_loss(y_all, clip_prob(p), labels=[0,1])),
        "incident_macro_brier": float(inc["brier"].mean()),
        "incident_macro_log_loss": float(inc["log_loss"].mean()),
        "mixed_incident_macro_pr_auc": float(inc.loc[mixed,"pr_auc"].mean()),
        "mixed_incident_macro_roc_auc": float(inc.loc[mixed,"roc_auc"].mean()),
    })

point_metrics = pd.DataFrame(point_rows)
display(point_metrics)

point_metrics.to_csv(
    OUTPUT_ROOT / "PRIMARY_MODEL_POINT_METRICS_V1.csv",
    index=False,
)

## 3. Paired incident-cluster bootstrap

The wildfire incident is the bootstrap unit. All models use the exact same
incident draw in each replicate, so differences are paired.

In [ ]:
incident_ids = np.array(sorted(pred["_irwin_norm"].unique()))
assert len(incident_ids) == EXPECTED_INCIDENTS

aligned = {
    model_id: incident_tables[model_id].set_index("_irwin_norm").loc[incident_ids]
    for model_id in MODEL_IDS
}

rng = np.random.default_rng(BOOTSTRAP_SEED)
boot_rows = []

for rep in range(BOOTSTRAP_REPS):
    draw = rng.integers(0, EXPECTED_INCIDENTS, size=EXPECTED_INCIDENTS)
    mult = np.bincount(draw, minlength=EXPECTED_INCIDENTS).astype(float)

    for model_id in MODEL_IDS:
        x = aligned[model_id]
        units = x["units"].to_numpy(float)
        total_units = float(np.sum(mult * units))
        mixed = x["mixed_class"].to_numpy(bool)
        mixed_mult = mult[mixed]

        boot_rows.append({
            "rep": rep,
            "model_id": model_id,
            "pooled_brier": float(np.sum(mult*x["brier_sum"].to_numpy(float))/total_units),
            "pooled_log_loss": float(np.sum(mult*x["logloss_sum"].to_numpy(float))/total_units),
            "incident_macro_brier": float(np.sum(mult*x["brier"].to_numpy(float))/np.sum(mult)),
            "incident_macro_log_loss": float(np.sum(mult*x["log_loss"].to_numpy(float))/np.sum(mult)),
            "mixed_incident_macro_pr_auc": float(
                np.sum(mixed_mult*x.loc[mixed,"pr_auc"].to_numpy(float))/np.sum(mixed_mult)
            ),
            "mixed_incident_macro_roc_auc": float(
                np.sum(mixed_mult*x.loc[mixed,"roc_auc"].to_numpy(float))/np.sum(mixed_mult)
            ),
        })

bootstrap = pd.DataFrame(boot_rows)

BOOT_METRICS = [
    "pooled_brier",
    "pooled_log_loss",
    "incident_macro_brier",
    "incident_macro_log_loss",
    "mixed_incident_macro_pr_auc",
    "mixed_incident_macro_roc_auc",
]

ci_rows = []
for model_id in MODEL_IDS:
    g = bootstrap[bootstrap["model_id"] == model_id]
    for metric in BOOT_METRICS:
        v = g[metric].to_numpy(float)
        ci_rows.append({
            "model_id": model_id,
            "metric": metric,
            "bootstrap_mean": float(v.mean()),
            "ci_low": float(np.quantile(v, 0.025)),
            "ci_high": float(np.quantile(v, 0.975)),
        })

bootstrap_ci = pd.DataFrame(ci_rows)
display(bootstrap_ci)

bootstrap_ci.to_csv(
    OUTPUT_ROOT / "INCIDENT_CLUSTER_BOOTSTRAP_CI_V1.csv",
    index=False,
)

## 4. Paired differences versus the frozen primary

In [ ]:
primary_boot = bootstrap[
    bootstrap["model_id"] == PRIMARY_MODEL_ID
].set_index("rep")

delta_rows = []

LOSS_METRICS = {
    "pooled_brier",
    "pooled_log_loss",
    "incident_macro_brier",
    "incident_macro_log_loss",
}

for alt in MODEL_IDS:
    if alt == PRIMARY_MODEL_ID:
        continue

    alt_boot = bootstrap[
        bootstrap["model_id"] == alt
    ].set_index("rep")

    assert alt_boot.index.equals(primary_boot.index)

    for metric in BOOT_METRICS:
        delta = (
            alt_boot[metric].to_numpy(float)
            - primary_boot[metric].to_numpy(float)
        )

        alt_better = (
            delta < 0
            if metric in LOSS_METRICS
            else delta > 0
        )

        delta_rows.append({
            "alternative_model": alt,
            "metric": metric,
            "delta_alt_minus_primary_mean": float(delta.mean()),
            "ci_low": float(np.quantile(delta,0.025)),
            "ci_high": float(np.quantile(delta,0.975)),
            "probability_alt_better_than_primary": float(np.mean(alt_better)),
        })

paired_deltas = pd.DataFrame(delta_rows)
display(paired_deltas)

paired_deltas.to_csv(
    OUTPUT_ROOT / "PAIRED_BOOTSTRAP_DELTAS_VS_PRIMARY_V1.csv",
    index=False,
)

## 5. Per-year and per-fold robustness

In [ ]:
def simple_metrics(frame, model_id):
    y = frame["hold_label"].to_numpy(int)
    p = frame[model_id].to_numpy(float)
    return {
        "units": len(frame),
        "incidents": frame["_irwin_norm"].nunique(),
        "held_rate": float(y.mean()),
        "mean_pred": float(p.mean()),
        "pr_auc": float(average_precision_score(y,p)),
        "roc_auc": float(roc_auc_score(y,p)),
        "brier": float(brier_score_loss(y,p)),
        "log_loss": float(log_loss(y, clip_prob(p), labels=[0,1])),
    }

year_rows = []
for year, g in pred.groupby("Year"):
    for model_id in MODEL_IDS:
        year_rows.append({
            "Year": year,
            "model_id": model_id,
            **simple_metrics(g, model_id),
        })

fold_rows = []
for fold, g in pred.groupby("outer_fold"):
    for model_id in MODEL_IDS:
        fold_rows.append({
            "outer_fold": fold,
            "model_id": model_id,
            **simple_metrics(g, model_id),
        })

year_metrics = pd.DataFrame(year_rows)
fold_metrics = pd.DataFrame(fold_rows)

display(year_metrics[year_metrics["model_id"] == PRIMARY_MODEL_ID])
display(fold_metrics[fold_metrics["model_id"] == PRIMARY_MODEL_ID])

year_metrics.to_csv(
    OUTPUT_ROOT / "PER_YEAR_METRICS_V1.csv",
    index=False,
)

fold_metrics.to_csv(
    OUTPUT_ROOT / "PER_OUTER_FOLD_METRICS_V1.csv",
    index=False,
)

## 6. Primary reliability and final freeze

In [ ]:
edges = np.linspace(0.0,1.0,11)

rel = pred[
    ["hold_label", PRIMARY_MODEL_ID]
].copy()

rel["bin"] = pd.cut(
    rel[PRIMARY_MODEL_ID],
    bins=edges,
    include_lowest=True,
)

reliability = (
    rel.groupby("bin", observed=True, as_index=False)
    .agg(
        units=("hold_label","size"),
        mean_pred=(PRIMARY_MODEL_ID,"mean"),
        observed_hold_rate=("hold_label","mean"),
    )
)

reliability["calibration_gap"] = (
    reliability["mean_pred"]
    - reliability["observed_hold_rate"]
)

display(reliability)

reliability.to_csv(
    OUTPUT_ROOT / "PRIMARY_MODEL_RELIABILITY_TABLE_V1.csv",
    index=False,
)

lookup = point_metrics.set_index("model_id")
primary_brier = float(
    lookup.loc[PRIMARY_MODEL_ID, "pooled_brier"]
)

checks = pd.DataFrame([
    {
        "check": "exact_304143_aligned_oof_units",
        "pass": len(pred) == EXPECTED_UNITS,
    },
    {
        "check": "exact_381_incidents",
        "pass": pred["_irwin_norm"].nunique() == EXPECTED_INCIDENTS,
    },
    {
        "check": "all_four_probability_sets_finite",
        "pass": bool(np.isfinite(pred[MODEL_IDS].to_numpy(float)).all()),
    },
    {
        "check": "paired_incident_bootstrap_5000_reps_complete",
        "pass": bootstrap["rep"].nunique() == BOOTSTRAP_REPS,
    },
    {
        "check": "primary_has_lowest_pooled_brier_among_compared_pipelines",
        "pass": bool(
            primary_brier
            <= lookup["pooled_brier"].min() + 1e-12
        ),
    },
    {
        "check": "no_model_fitting_in_05e",
        "pass": True,
    },
    {
        "check": "primary_model_frozen_as_05c_fixed_row_raw",
        "pass": PRIMARY_MODEL_ID == "05C_FIXED_ROW_RAW",
    },
])

display(checks)

PASS = bool(checks["pass"].all())

print()
print(
    "05E PRIMARY MODEL FREEZE:",
    (
        "PASS / PRIMARY MODEL FROZEN; READY FOR FINAL FULL-DATA FIT"
        if PASS
        else "REVIEW REQUIRED"
    ),
)

checks.to_csv(
    OUTPUT_ROOT / "FINAL_PRIMARY_MODEL_FREEZE_CHECKS_V1.csv",
    index=False,
)

## 7. Final report

In [ ]:
def tomd(df):
    try:
        return df.to_markdown(index=False)
    except Exception:
        return df.to_string(index=False)

report = "\n".join([
    "# 05E Primary Model Freeze and Incident-Level Uncertainty Audit",
    "",
    f"Build: `{NOTEBOOK_BUILD_ID}`",
    "",
    "## Frozen primary model",
    "",
    f"`{PRIMARY_MODEL_ID}`",
    "",
    "## Point metrics",
    "",
    tomd(point_metrics),
    "",
    "## Incident-cluster bootstrap confidence intervals",
    "",
    tomd(bootstrap_ci),
    "",
    "## Paired bootstrap deltas versus primary",
    "",
    tomd(paired_deltas),
    "",
    "## Primary model per year",
    "",
    tomd(year_metrics[year_metrics["model_id"] == PRIMARY_MODEL_ID]),
    "",
    "## Primary model per outer fold",
    "",
    tomd(fold_metrics[fold_metrics["model_id"] == PRIMARY_MODEL_ID]),
    "",
    "## Reliability table",
    "",
    tomd(reliability),
    "",
    "## Final checks",
    "",
    tomd(checks),
    "",
    "## Decision",
    "",
    (
        "`PASS / PRIMARY MODEL FROZEN; READY FOR FINAL FULL-DATA FIT`"
        if PASS
        else "`REVIEW REQUIRED`"
    ),
])

REPORT_PATH = (
    OUTPUT_ROOT / "PRIMARY_MODEL_FREEZE_REPORT_V1.md"
)

REPORT_PATH.write_text(
    report,
    encoding="utf-8",
)

print(report)
print()
print("Saved:", REPORT_PATH)